<a href="https://colab.research.google.com/github/Maxxx-VS/IMA_SibADI/blob/main/ML_1_1_%D0%BE%D0%BF%D1%82%D0%B8%D0%BC%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F_%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D0%BE%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 ### Задание 1.1. Оптимизация параметров линейной регрессии

In [14]:
# импорты
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import plotly.graph_objects as go

In [15]:
# загрузка данных
df = pd.read_csv('/content/data_for_regression.csv')

In [16]:
# извлечение x, y
x = df.Me.to_numpy()
y = df.tc.to_numpy()
print("x = ", x, "\n", "y = ", y)

x =  [140 210 252  28  70 140 210 252  28  70 140 210 252  28  70 140 210 252
  28  70 140 210] 
 y =  [80.7 82.8 83.2 79.3 80.2 80.7 83.2 83.7 80.3 80.4 81.5 82.3 84.3 80.2
 80.7 82.1 83.1 85.  80.7 80.8 82.8 83.6]


In [17]:
# визуализация исходных данных
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode='markers',
    name='исходные данные',
    marker=dict(color='magenta', size=7),
    opacity=0.8))
fig.update_layout(
    title_text="Температура в зависимости от крутящего момента",
    title_font_size=20,
    xaxis_title="x",
    yaxis_title="y")
fig.show()

In [18]:
# МНК (OLS)
X = sm.add_constant(x)
reg = sm.OLS(y, X).fit()
b_ols = reg.params[0]
w_ols = reg.params[1]
print("Оптимальные параметры регрессии: b = %0.4f, w = %0.4f \n" % (b_ols, w_ols))

Оптимальные параметры регрессии: b = 79.3802, w = 0.0175 



In [19]:
# расчетные значения
y_hat = reg.fittedvalues
print("Расчетные (прогнозные) значения:\n", y_hat)

Расчетные (прогнозные) значения:
 [81.83511644 83.06255482 83.79901785 79.87121502 80.60767805 81.83511644
 83.06255482 83.79901785 79.87121502 80.60767805 81.83511644 83.06255482
 83.79901785 79.87121502 80.60767805 81.83511644 83.06255482 83.79901785
 79.87121502 80.60767805 81.83511644 83.06255482]


In [20]:
# модель на графике
fig.add_trace(
    go.Scatter(
        x=x, y=y_hat,
        mode='lines+markers',
        name='модель МНК',
        marker=dict(color='green', size=7),
        opacity=0.8))
fig.show()

In [21]:
# остатки
e = y - (b_ols + w_ols * x)
print("Остатки:\n", e)

Остатки:
 [-1.13511644 -0.26255482 -0.59901785 -0.57121502 -0.40767805 -1.13511644
  0.13744518 -0.09901785  0.42878498 -0.20767805 -0.33511644 -0.76255482
  0.50098215  0.32878498  0.09232195  0.26488356  0.03744518  1.20098215
  0.82878498  0.19232195  0.96488356  0.53744518]


In [22]:
# MSE
loss_ols = (e ** 2).mean()
print("Минимальное значение функции потерь MSE: %0.4f" % loss_ols)

Минимальное значение функции потерь MSE: 0.3760


In [23]:
# пространство параметров
b_range = np.linspace(b_ols * 0.95, b_ols * 1.05, 201)
w_range = np.linspace(w_ols * 0.7, w_ols * 1.3, 201)
bs, ws = np.meshgrid(b_range, w_range)
bs.shape, ws.shape

((201, 201), (201, 201))

In [24]:
# изменение формы x
x_reshaped = x.reshape((-1, 1))
x_reshaped

array([[140],
       [210],
       [252],
       [ 28],
       [ 70],
       [140],
       [210],
       [252],
       [ 28],
       [ 70],
       [140],
       [210],
       [252],
       [ 28],
       [ 70],
       [140],
       [210],
       [252],
       [ 28],
       [ 70],
       [140],
       [210]])

In [25]:
# все возможные выходы модели
all_y_hat = np.apply_along_axis(func1d=lambda x: bs + ws * x,
                                axis=1,
                                arr=x_reshaped)
all_y_hat.shape

(22, 201, 201)

In [26]:
# изменение формы y
y_reshaped = y.reshape(-1, 1, 1)
y_reshaped.shape

(22, 1, 1)

In [27]:
# ошибки на сетке
all_errors = all_y_hat - y_reshaped
all_errors.shape

(22, 201, 201)

In [28]:
# MSE на сетке
all_losses = (all_errors ** 2).mean(axis=0)
all_losses.shape

(201, 201)

In [29]:
# контурный график MSE
fig2 = go.Figure()
fig2.add_trace(go.Contour(x=b_range,
                          y=w_range,
                          z=all_losses,
                          contours_coloring='lines',
                          line_width=1,
                          name='MSE',
                          contours=dict(start=0, end=14, size=0.2,
                                        showlabels=True,
                                        labelfont=dict(size=12, color='black')),
                          showlegend=True, showscale=False))

fig2.add_trace(go.Scatter(x=[b_ols], y=[w_ols],
                          mode='markers',
                          name='Оптимальные параметры',
                          marker=dict(color='green', size=7, opacity=0.8),
                          showlegend=True))

fig2.update_layout(title_text="Критерий точности модели: Функция потерь",
                   title_font_size=16,
                   xaxis_title="b", yaxis_title="w")
fig2.show()

In [30]:
# поверхность MSE в 3D
fig3 = go.Figure()
fig3.add_trace(go.Scatter3d(x=bs.flatten(), y=ws.flatten(), z=all_losses.flatten(),
                            mode='markers',
                            name='MSE',
                            marker=dict(size=1, color='blue', opacity=0.8)))
fig3.add_trace(go.Scatter3d(x=[b_ols], y=[w_ols], z=[loss_ols],
                            mode='markers',
                            name='Оптимальные параметры',
                            marker=dict(size=7, color='green', opacity=0.8)))
fig3.update_layout(
    scene=dict(xaxis_title='b', yaxis_title='w', zaxis_title='MSE'),
    title_text="Критерий точности модели: Функция потерь",
    title_font_size=16)
fig3.layout.scene.camera.projection.type = "orthographic"
fig3.show()

In [31]:
# стартовые параметры
b0 = 78
w0 = 0.018

In [32]:
# MSE в стартовой точке
x_reshaped = x.flatten()
y_reshaped = y.flatten()
e = (b0 + w0 * x_reshaped) - y_reshaped
loss = (e ** 2).mean()
print(loss)

2.1030567272727207


In [33]:
# визуализация старта
fig2.add_trace(go.Scatter(
    x=[b0], y=[w0],
    mode='markers',
    name='Старт поиска (b0, w0)',
    marker=dict(color='red', size=7, opacity=0.8),
    showlegend=True))
fig2.show()

In [34]:
# градиент
b_grad = 2 * e.mean()
w_grad = 2 * (x * e).mean()
print("Градиент функции потерь: [%0.2f %0.2f]" % (b_grad, w_grad))

Градиент функции потерь: [-2.63 -370.08]


In [35]:
# векторы градиента и антиградиента
fig2.add_trace(
    go.Scatter(x=[b0, b0 + b_grad * 0.00001],
               y=[w0, w0 + w_grad * 0.00001],
               name='градиент * 0.00001',
               marker=dict(color='blue', size=7,
                           symbol="arrow-bar-up", angleref="previous"),
               showlegend=True))
fig2.add_trace(
    go.Scatter(x=[b0, b0 - b_grad * 0.00001],
               y=[w0, w0 - w_grad * 0.00001],
               name='антиградиент * 0.00001',
               marker=dict(color='red', size=7,
                           symbol="arrow-bar-up", angleref="previous"),
               showlegend=True))
fig2.show()

In [36]:
# одинаковый масштаб осей
fig2.update_yaxes(scaleanchor="x", scaleratio=1)
fig2.show()

In [37]:
# увеличение фрагмента
fig2.update_layout(xaxis=dict(range=[77.98, 78.04]))
fig2.show()

In [38]:
# learning rate
lr = 0.00002

In [39]:
# шаг 1 оптимизации
b1 = b0 - lr * b_grad
w1 = w0 - lr * w_grad
print("Параметры b и w после шага 1: [%0.2f %0.5f]" % (b1, w1))

Параметры b и w после шага 1: [78.00 0.02540]


In [40]:
# MSE после шага 1
e = (b1 + w1 * x_reshaped) - y_reshaped
loss = (e ** 2).mean()
print(loss)

0.8461329218777812


In [41]:
# новый градиент после шага 1
b_grad = 2 * e.mean()
w_grad = 2 * (x * e).mean()
print("Градиент функции потерь: [%0.2f %0.2f]" % (b_grad, w_grad))

Градиент функции потерь: [-0.51 30.46]


In [42]:
# визуализация после шага 1
fig2.add_trace(go.Scatter(
    x=[b1], y=[w1],
    mode='markers',
    name='После шага 1: (b1, w1)',
    marker=dict(color='black', size=7, opacity=0.8),
    showlegend=True))

fig2.add_trace(go.Scatter(
    x=[b1, b1 - b_grad * 0.0001],
    y=[w1, w1 - w_grad * 0.0001],
    name='антиградиент * 0.0001 на шаге 1',
    marker=dict(color='black', size=7,
                symbol="arrow-bar-up", angleref="previous"),
    showlegend=True))
fig2.show()

In [43]:
# шаг 2
b2 = b1 - lr * b_grad
w2 = w1 - lr * w_grad
print("Параметры b и w после шага 2: [%0.2f %0.5f]" % (b2, w2))

e = (b2 + w2 * x_reshaped) - y_reshaped
loss = (e ** 2).mean()
print("MSE после шага 2:", loss)

b_grad = 2 * e.mean()
w_grad = 2 * (x * e).mean()
print("Градиент функции потерь: [%0.2f %0.2f]" % (b_grad, w_grad))

fig2.add_trace(go.Scatter(
    x=[b2], y=[w2],
    mode='markers',
    name='После шага 2: (b2, w2)',
    marker=dict(color='orange', size=7, opacity=0.8),
    showlegend=True))
fig2.add_trace(go.Scatter(
    x=[b2, b2 - b_grad * 0.0001],
    y=[w2, w2 - w_grad * 0.0001],
    name='антиградиент * 0.0001 на шаге 2',
    marker=dict(color='orange', size=7,
                symbol="arrow-bar-up", angleref="previous"),
    showlegend=True))
fig2.show()

Параметры b и w после шага 2: [78.00 0.02479]
MSE после шага 2: 0.8376091460235485
Градиент функции потерь: [-0.68 -2.50]


In [44]:
# шаг 3
b3 = b2 - lr * b_grad
w3 = w2 - lr * w_grad
print("Параметры b и w после шага 3: [%0.2f %0.5f]" % (b3, w3))

e = (b3 + w3 * x_reshaped) - y_reshaped
loss = (e ** 2).mean()
print("MSE после шага 3:", loss)

b_grad = 2 * e.mean()
w_grad = 2 * (x * e).mean()
print("Градиент функции потерь: [%0.2f %0.2f]" % (b_grad, w_grad))

fig2.add_trace(go.Scatter(
    x=[b3], y=[w3],
    mode='markers',
    name='После шага 3: (b3, w3)',
    marker=dict(color='purple', size=7, opacity=0.8),
    showlegend=True))
fig2.add_trace(go.Scatter(
    x=[b3, b3 - b_grad * 0.0001],
    y=[w3, w3 - w_grad * 0.0001],
    name='антиградиент * 0.0001 на шаге 3',
    marker=dict(color='purple', size=7,
                symbol="arrow-bar-up", angleref="previous"),
    showlegend=True))
fig2.show()

Параметры b и w после шага 3: [78.00 0.02484]
MSE после шага 3: 0.8375425168251183
Градиент функции потерь: [-0.67 0.21]


In [45]:
# сравнение с оптимумом
print("Оптимум OLS: b = %0.4f, w = %0.4f, MSE = %0.4f" % (b_ols, w_ols, loss_ols))
print("После 3 шагов: b = %0.4f, w = %0.5f, MSE = %0.4f" % (b3, w3, loss))
print("Вывод: без стандартизации x движение к оптимуму по b очень медленное.")

Оптимум OLS: b = 79.3802, w = 0.0175, MSE = 0.3760
После 3 шагов: b = 78.0001, w = 0.02484, MSE = 0.8375
Вывод: без стандартизации x движение к оптимуму по b очень медленное.
